# Encrypted configuration manager

Run the setup and helper-definition cells first. They do not read, print, or change your configuration. Then call the commands you need.

The complete config document is encrypted on disk, including its field names and values. Decryption happens in Python memory. The key comes from `LRAEP_CONFIG_KEY`, if set, or the separate `config.json.key` file next to the config. Keep that key private and backed up; losing it makes the config unreadable.

Select the **LRAEP trainer** kernel, or choose this project's `.venv/bin/python` interpreter in VS Code. This kernel needs Python 3.12+ and the project dependencies. From the project folder, install dependencies with:

```bash
uv pip install --python .venv/bin/python -r requirements.txt
```

If the named kernel is not registered on another machine, register it from the project folder, then select **LRAEP trainer**:

```bash
uv pip install --python .venv/bin/python ipykernel
.venv/bin/python -m ipykernel install --user --name lraep-trainer --display-name 'LRAEP trainer'
```

**Plaintext display is explicit.** Calling `show_config()` prints secrets into the cell output. Notebook editors may autosave that output. Clear all outputs before saving or sharing this notebook, and restart the kernel when finished. Setup disables IPython command-history logging for this kernel; this does not disable notebook source or output saving. Avoid typing real secrets directly into saved cells; the `getpass` example below accepts a value without echoing or placing it in cell source.


In [ ]:
import json
import sys
from getpass import getpass
from pathlib import Path

# Run this cell before entering secrets. Do not record later commands or output
# in the IPython history database. Notebook output saving is separate.
try:
    shell = get_ipython()
except NameError:
    shell = None
if shell is not None:
    shell.history_manager.enabled = False
    shell.history_manager.db_log_output = False


def find_project_root():
    here = Path.cwd().resolve()
    for directory in (here, *here.parents):
        for candidate in (directory, directory / "trainer"):
            if (candidate / "ann_trainer.py").is_file() and (
                candidate / "src" / "LRAEP_trainer" / "config.py"
            ).is_file():
                return candidate
    raise RuntimeError("Open this notebook from the trainer project or its parent folder.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from LRAEP_trainer.config import (
    encrypt_config_file,
    read_config,
    save_config,
    update_config,
)
from LRAEP_trainer.config_crypto import decode_string, encode_string, generate_key

CONFIG_PATH = PROJECT_ROOT / "config.json"  # Change this to manage another file.
print(f"Ready to manage {CONFIG_PATH}; no config values have been loaded.")


In [ ]:
def show_config():
    """Decrypt and print the original config only when explicitly called."""
    print(json.dumps(read_config(CONFIG_PATH), indent=2, ensure_ascii=False))


def update_from_dict(plain_dict):
    """Deep-merge a plain dictionary, validate it, and save encrypted."""
    update_config(CONFIG_PATH, plain_dict)
    print("Encrypted configuration updated.")


def replace_from_dict(plain_dict):
    """Validate and replace the complete config with an encrypted dictionary."""
    save_config(CONFIG_PATH, plain_dict)
    print("Encrypted configuration replaced.")


def encrypt_existing_config():
    """Migrate an existing plain JSON config without printing its contents."""
    encrypt_config_file(CONFIG_PATH)
    print("Configuration stored encrypted.")


## Print on command

Uncomment and run `show_config()` below to decrypt and print the config. The file remains encrypted. Clear the printed output before saving this notebook.


In [ ]:
# show_config()


## Update from a plain dictionary

`update_from_dict` recursively merges nested dictionaries, preserving fields you did not supply. It validates the resulting configuration before writing. The saved file contains only the encrypted document and format metadata.

The following harmless example is prepared but does not save anything until you uncomment the last line. For complete replacement, pass a full dictionary including `"version": 1` to `replace_from_dict`. Relative paths stay relative to the config file.


In [ ]:
plain_dict = {
    "pool": {"worker": "Rig01"},
    "training": {"max_batches": 0},
}

# update_from_dict(plain_dict)


## Enter a secret without putting it in cell source

Run these commands only when you want to change the pool password. `getpass` hides the input. The resulting dictionary remains in kernel memory, so delete it afterward and restart the kernel when done. Do not display the dictionary.


In [ ]:
# plain_dict = {"pool": {"password": getpass("New pool password: ")}}
# update_from_dict(plain_dict)
# del plain_dict


## Optional migration and string helpers

For a new plaintext config, call `encrypt_existing_config()` once. A missing local key is created with owner-only permissions on the first encrypted save or migration. Reads never create a replacement key. Public `config.example.json` remains a plaintext template.

The string helpers accept an explicit Fernet key, or use `LRAEP_CONFIG_KEY` when no key is passed. They do not implicitly use `config.json.key`. Generate a key in memory without displaying it, then pass the same key to both operations.


In [ ]:
# encrypt_existing_config()

# key = generate_key()
# encoded = encode_string("example string", key=key)
# decoded = decode_string(encoded, key=key)
# assert decoded == "example string"
# del decoded, encoded, key
